In [ ]:
from transformers import pipeline
import ray
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer
from PIL import Image
import os

## Scaling up and launching a batch job

Based on the performance of our data subset, we estimate that running the full 1000 records on 2xA10G will take about 20 minutes. 

Let's...

1. check linear scaling: see if we can cut this down to 5 minutes by using 8 GPUs
2. set up an Anyscale batch job to do this, which is how we'll run in production

To run a job, we usually need two parts:

1. a YAML file that describes our cluster (hardware and scaling), our software (container image and/or dependencies), environment vars, other metadata ... and the Python script to run
2. that Python script containing our Ray code

Here is a a YAML file for the job we want (along with some extra bits for illustrative purposes)

```yaml
#create_extended_desc_job.yaml

name: create_extended_desc_job
entrypoint: python create_extended_desc_job.py

# Container image: Can be an Anyscale base image, external registry, or custom image
image_uri: anyscale/image/c26:2
# OR use containerfile: ./Dockerfile (see "Using a custom container" section below)

# Compute config: Can be name of existing config OR inline definition
# Use existing: compute_config: my-compute-config:1
# Or define inline as shown below
compute_config: 
  head_node:
    instance_type: m5.2xlarge
  worker_nodes:
    - instance_type: g5.2xlarge
      min_nodes: 8
      max_nodes: 8

working_dir: . # Local directory to upload (defaults to current directory)
#requirements: # Python dependencies - can be list or path to requirements.txt
#  - numpy==1.26.4
#  - pandas==2.3.3
env_vars:
  HF_TOKEN: hf_monkeyphonics
max_retries: 3
tags:
  team: training-26
```

For our script, we can *mostly* use the code that we (and AI) have written ... but we'd like the Actor scaling to adjust if we use a larger or smaller cluster, and our Actor count is based on the number of GPUs available. We can detect the GPU count and parametrize in our code:

In [ ]:
ray.init()

ray.cluster_resources()

In [ ]:
int(ray.cluster_resources()['GPU'])

We also need to check our storage locations. `/mnt/cluster_storage` is great ... But jobs get their own ephemeral clusters (in standard cloud-native fashion) so they can't read the `cluster_storage` we have been writing to ... and their outpus shouldn't go to `cluster_storage` since that dies along with them. 

We can write to `user_storage` instead. A cloud blobstore bucket is also a common alternative.

In [ ]:
%%sh

rm -rf /mnt/user_storage/cat_with_embeddings/

cp -r /mnt/cluster_storage/cat_with_embeddings/ /mnt/user_storage/

```python
from transformers import pipeline
import ray
import torch
import os

MODEL_NAME = "/mnt/shared_storage/course_assets/ecom/hf_cache/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767"
IMAGE_DIR = "/mnt/shared_storage/course_assets/ecom/catalog_images/"
INPUT_DIR = "/mnt/user_storage/cat_with_embeddings/" # <- needs to exist outside of the job's cluster
OUTPUT_PATH = "/mnt/user_storage/cat_with_embed_and_extended_desc.parquet" # <- we want this to exist after the job
BATCH_SIZE = 8 

system_prompt = '''You are a helpful assistant. Given a product name and description, and an image of that product, 
please create a more descriptive and attractive blurb for the product, 
capturing elements from the image and suitable for use in an ecommerce website where that product is for sale.
Output a single suggestion or option, and do not include any additional conversational language or discussion.
'''

class VLMPredictor:
    def __init__(self):
        self.pipe = pipeline("image-text-to-text", model=MODEL_NAME, device="cuda:0")

    def __call__(self, batch: dict) -> dict:
        results = []
        messages = []

        for item_id, desc in zip(batch["item_id"], batch["desc"]):
            image_path = os.path.join(IMAGE_DIR, f"{item_id}.png")
            content = []
            content.append({"type": "image", "path": image_path})
            content.append({"type": "text", "text": desc})

            messages.append([
                { "role": "system", "content": [{"type": "text", "text": system_prompt}] },
                {"role": "user", "content": content}
            ])
            
        outputs = self.pipe(text=messages, max_new_tokens=1024, batch_size=len(messages))          
        batch["cat_desc"] = [out[0]["generated_text"][-1]["content"] for out in outputs]
        return batch

ray.init()
GPUs = int(ray.cluster_resources()['GPU'])

ray.data.read_parquet(INPUT_DIR).repartition(2*GPUs
    ).map_batches(
        VLMPredictor,
        batch_size=BATCH_SIZE,
        compute=ray.data.ActorPoolStrategy(size=GPUs),
        num_gpus=1,
    ).write_parquet(OUTPUT_PATH, mode=ray.data.SaveMode.OVERWRITE)

print(f'Catalog with extended descriptions added at {OUTPUT_PATH}')
```

Launch the job from the CLI (make sure to swap in your HF key first; in production, you'd probably access the key from a secret store or else ensure it's injected into the target cluster)

In [ ]:
! anyscale job submit --config-file create_extended_desc_job.yaml

The linear scaling appears to work. Let's check the output.

In [ ]:
ray.data.read_parquet('/mnt/user_storage/cat_with_embed_and_extended_desc.parquet').count()

In [ ]:
ray.data.read_parquet('/mnt/user_storage/cat_with_embed_and_extended_desc.parquet').take(1)